#BASIC

In [0]:
# 1
employee = [
    {"Name": "Alice", "Age": 25, "Dept": "HR"},
    {"Name": "Bob", "Age": 30, "Dept": "IT"},
    {"Name": "Charlie", "Age": 35, "Dept": "Finance"}
]
employee_df = spark.createDataFrame(employee)

display(employee_df)

Age,Dept,Name
25,HR,Alice
30,IT,Bob
35,Finance,Charlie


In [0]:
# 2
sales= spark.read.csv("/Volumes/dev/demo/raw-1000-richest/sales.csv", header=True, inferSchema=True)

sales.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- order_date: date (nullable = true)



In [0]:
# Read the CSV file with define explicit schema
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField("order_id", IntegerType(), True),     
    StructField("customer_id", IntegerType(), True),  
    StructField("transaction_id", IntegerType(), True), 
    StructField("product_id", IntegerType(), True),     
    StructField("quantity", IntegerType(), True),      
    StructField("discount_amount", DoubleType(), True), 
    StructField("total_amount", DoubleType(), True),   
    StructField("order_date", DateType(), True)
])

sales2 = spark.read.csv("/Volumes/dev/demo/raw-1000-richest/sales.csv", header=True, schema=schema)

sales2.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- order_date: date (nullable = true)



In [0]:
# 3
from pyspark.sql.functions import col
sales.filter(col("total_amount") > 1000).show()

#the reason behind this is that pysaprk works on lazy eveluation concept.it doesn't execute the query until the action is called.

+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|order_id|customer_id|transaction_id|product_id|quantity|discount_amount|total_amount|order_date|
+--------+-----------+--------------+----------+--------+---------------+------------+----------+
|       1|          1|          1001|         3|       3|           9.02|     1300.12|2023-12-02|
|      15|          3|          1015|         2|       5|          15.61|     1164.44|2023-12-05|
|      24|          2|          1024|         3|       3|           3.67|     1305.47|2023-12-04|
|      27|          5|          1027|         3|       4|          18.17|     1727.35|2023-12-05|
|      32|          2|          1032|         3|       5|           6.58|     2175.32|2023-12-05|
|      36|          1|          1036|         3|       4|           8.57|     1736.95|2023-12-01|
|      39|          1|          1039|         5|       5|           1.64|     1008.21|2023-12-02|
|      41|          

#INTERMEDIATE

In [0]:
# 4.
from pyspark.sql.functions import current_date

sales_df = spark.read.csv("/Volumes/dev/demo/raw-1000-richest/sales.csv", header=True, inferSchema=True)

sales_df = sales_df.filter("total_amount > 1000")

sales_df = sales_df.withColumn("ingestion_date" , current_date())

sales_df.write.mode("overwrite").saveAsTable("sales")



In [0]:
# 5.
from pyspark.sql.functions import col

df = spark.read.json("/Volumes/dev/demo/raw-1000-richest/drivers.json")
flattened_df = df.select(
    "driverId",
    "code",
    col("name.forename").alias("first_name"),
    col("name.surname").alias("last_name"),
)

display(flattened_df)


driverId,code,first_name,last_name
1,HAM,Lewis,Hamilton
2,HEI,Nick,Heidfeld
3,ROS,Nico,Rosberg
4,ALO,Fernando,Alonso
5,KOV,Heikki,Kovalainen
6,NAK,Kazuki,Nakajima
7,BOU,Sébastien,Bourdais
8,RAI,Kimi,Räikkönen
9,KUB,Robert,Kubica
10,GLO,Timo,Glock


In [0]:
# 6

df = spark.read.csv(
    "/Volumes/dev/demo/raw-1000-richest/sales.csv",
    header=True,
    inferSchema=True
)

result = (
    df
    .filter(col("total_amount") > 1000)
    .select("customer_id", "total_amount", "product_id")
    .groupBy("customer_id")
    .sum("total_amount")
    .orderBy("sum(total_amount)", ascending=False)
)

result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [sum(total_amount)#11590 DESC NULLS LAST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7484]
               +- PhotonShuffleExchangeSink rangepartitioning(sum(total_amount)#11590 DESC NULLS LAST, 16)
                  +- PhotonGroupingAgg(keys=[customer_id#11579], functions=[finalmerge_sum(merge sum#11592) AS sum(total_amount)#11589])
                     +- PhotonShuffleExchangeSource
                        +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7478]
                           +- PhotonShuffleExchangeSink hashpartitioning(customer_id#11579, 16)
                              +- PhotonGroupingAgg(keys=[customer_id#11579], functions=[partial_sum(total_amount#11584) AS sum#11592])
                                 +- PhotonFilter (isnotnull(total_amount#11584) AND (total_amoun

#ADVANCED 

In [0]:
#7
import pandas as pd

df = pd.read_csv("/Volumes/dev/demo/raw-1000-richest/sales.csv")
df["order_date"] = pd.to_datetime(df["order_date"])
df["revenue"] = df["quantity"] * df["total_amount"]
df = df.dropna(subset=["order_id", "total_amount", "product_id"])

monthly_revenue = (
    df.groupby(
        [df["order_date"].dt.to_period("M"), "product_id"]
    )["revenue"]
    .sum()
    .reset_index()
)

monthly_revenue.columns = [
    "month",
    "product_id",
    "total_revenue"
]

print(monthly_revenue)

     month  product_id  total_revenue
0  2023-12         1.0       12248.62
1  2023-12         2.0       25377.82
2  2023-12         3.0       57635.04
3  2023-12         4.0       23798.11
4  2023-12         5.0       30746.85


In [0]:
from pyspark.sql import functions as F

sales_df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/dev/demo/raw-1000-richest/sales.csv"
)

sales_df = sales_df.withColumn("order_date", F.to_date("order_date"))
sales_df = sales_df.withColumn("revenue", F.col("quantity") * F.col("total_amount"))
sales_df = sales_df.dropna(subset=["order_id", "total_amount", "product_id"])

monthly_revenue = (
    sales_df.withColumn("month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("month", "product_id")
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy("month", "product_id")
)

monthly_revenue.show()

+-------+----------+------------------+
|  month|product_id|     total_revenue|
+-------+----------+------------------+
|2023-12|         1|12248.619999999999|
|2023-12|         2|          25377.82|
|2023-12|         3|          57635.04|
|2023-12|         4|          23798.11|
|2023-12|         5|          30746.85|
+-------+----------+------------------+




- Data doesn't fit RAM - Pandas loads the DataFrame into the driver's memory  while Spark distributes data across multiple workers.    
 - GroupBy becomes expensive -  Pandas performs the operation on one machine while Spark distributes the aggregation using a shuffle.
- Large CSV processing - One machine reads and processes the file  while Spark can read files in parallel.

-  Large joins -  Pandas needs sufficient memory for both DataFrames while Spark distributes the join across executors



In [0]:
# 8

from pyspark.sql import functions as F

sales_df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/dev/demo/raw-1000-richest/sales.csv"
)
sales_df = (
    sales_df
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
)


the sales table is mostly queried by date range, I would partition it by year and month using partitionBy("year", "month"). 
I would avoid high-cardinality columns such as customer_id because they can create too many small partitions. I would target reasonably sized files, approximately 128 MB–1 GB. Spark transformations are lazy, so a filter does not immediately execute. When an action such as show() or count() runs, Spark creates a physical plan or perform action.Spark can perform partition pruning and read only the relevant date partitions. 

In [0]:
# 9 
from pyspark.sql import functions as F

df = spark.read.option("header", True).option("inferSchema", True).csv(
    "/Volumes/dev/demo/raw-1000-richest/sales.csv"
)
df = df.withColumn("order_date", F.to_date("order_date"))
df = df.withColumn("month", F.date_format("order_date", "yyyy-MM"))
df = df.dropna()
df = df.dropDuplicates()
monthly_revenue = (
    df
    .groupBy("month", "product_id")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy("month", "product_id")
)
monthly_revenue.show()

+-------+----------+------------------+--------------+------------+
|  month|product_id|     total_revenue|total_quantity|total_orders|
+-------+----------+------------------+--------------+------------+
|2023-12|         1|            2153.8|            25|           7|
|2023-12|         2|5112.5599999999995|            22|           7|
|2023-12|         3|          13000.27|            30|           9|
|2023-12|         4| 5198.640000000001|            43|          14|
|2023-12|         5|           6164.09|            31|          13|
+-------+----------+------------------+--------------+------------+



In [0]:
monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cyntexa_dev.sales.monthly_revenue")